In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("final_project_reviews") \
    .getOrCreate()

print(spark)

In [ ]:
# import sys
# print(sys.version)
# print(spark.version)

In [ ]:
# import os
# import subprocess
# import datetime
# import pandas as pd

# import matplotlib.pyplot as plt
# import seaborn as sns

# from pyspark.sql import functions as F
# from pyspark.sql.types import *

# pd.set_option('display.max_rows', 100)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_colwidth', None)

In [ ]:
gcs_folder = "data/reviews-source"

df_reviews = spark.read.parquet(f"{gcs_folder}/reviews_parquet")
df_meta    = spark.read.parquet(f"{gcs_folder}/meta_parquet")

df_reviews.printSchema()
df_meta.printSchema()

In [ ]:
df_reviews.count()
df_meta.count()


In [ ]:
sample_reviews = df_reviews.limit(50000)
sample_meta = df_meta.limit(50000)

display(sample_reviews)
display(sample_meta)


In [ ]:
sample_reviews.select("timestamp").show(5, truncate=False)


In [ ]:
from pyspark.sql.functions import from_unixtime, to_date, col

sample_reviews = sample_reviews.withColumn(
    "review_date",
    to_date(from_unixtime((col("timestamp")/1000).cast("long")))
)


In [ ]:
sample_reviews.selectExpr(
    "min(review_date) as min_date",
    "max(review_date) as max_date"
).show()


In [ ]:
sample_meta.select("categories").show(5, truncate=False)


In [ ]:
from pyspark.sql.functions import element_at, col

sample_meta = sample_meta.withColumn(
    "main_category_clean",
    element_at(col("categories"), 1)
)


In [ ]:
sample_meta.select("categories", "main_category_clean").show(5, truncate=False)


In [ ]:
sample = sample_reviews.join(
    sample_meta.select(
        "parent_asin",
        "main_category_clean",
        "price"
    ),
    on="parent_asin",
    how="left"
)


In [ ]:
sample.select(
    "parent_asin",
    "rating",
    "title",
    "main_category_clean",
    "price",
    "review_date"
).show(5, truncate=False)


In [ ]:
from pyspark.sql.functions import count

timeline = (
    sample.groupBy("review_date")
          .agg(count("*").alias("n_reviews"))
          .orderBy("review_date")
)

timeline.show(10)


In [ ]:
from pyspark.sql.functions import col


In [ ]:
sample_clean = (
    sample
    .filter(col("review_date").isNotNull())
    .filter(col("rating").between(1, 5))
    .filter(col("text").isNotNull())
)


In [ ]:
sample_clean = sample_clean.filter(col("review_date") >= "2004-01-01")


In [ ]:
sample_clean.count()


In [ ]:
from pyspark.sql.functions import from_unixtime, to_date, element_at

reviews_clean = (
    df_reviews
    .withColumn("review_date", to_date(from_unixtime(col("timestamp"))))
    .filter(col("review_date").isNotNull())
    .filter(col("rating").between(1, 5))
    .filter(col("text").isNotNull())
)

meta_clean = (
    df_meta
    .withColumn("main_category_clean", element_at(col("categories"), 1))
    .select("parent_asin", "main_category_clean", "price")
)

full_clean = reviews_clean.join(
    meta_clean,
    on="parent_asin",
    how="left"
)


In [ ]:
output_path = "data/spark-output/cleaned_reviews"

full_clean.write.mode("overwrite").parquet(output_path)
